# ClinicalBridge — RAG Pipeline

Demonstrates the Retrieval-Augmented Generation pipeline: EHR JSON → chunked text → embeddings → ChromaDB → similarity retrieval.

In [1]:
import os
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from pathlib import Path
import json

## 1. EHR Text Conversion

Raw EHR JSON is converted to human-readable text before chunking. This preserves clinical semantics better than JSON string embeddings.

In [2]:
from rag.ingest import ehr_to_text

pt001 = json.loads(Path("../data/patients/PT-001.json").read_text())
text = ehr_to_text(pt001)
print(f"Input: PT-001 EHR JSON")
print(f"Output text length: {len(text)} chars\n")
print("--- EHR Text ---")
print(text)

/Users/ayhan/Dersler/Prompt/Project/Code/clinicalbridge-capstone-main/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Input: PT-001 EHR JSON
Output text length: 1319 chars

--- EHR Text ---
Patient: Robert Harmon, age 64, sex M
Problem list: Essential hypertension (I10); Type 2 diabetes mellitus (E11.9); Hyperlipidemia (E78.5)
Medications: Lisinopril 10mg QD; Metformin 500mg BID; Atorvastatin 20mg QHS
Allergies: Penicillin - Urticaria
Lab BMP on 2025-10-15: Na=139, K=4.2, Cr=1.0, BUN=18, Glucose=112
Lab HbA1c on 2025-10-15: HbA1c=7.1
Lab Lipid panel on 2025-10-15: LDL=98, HDL=45, TG=142
Visit note 2025-10-15: S: Patient presents for routine follow-up. Reports occasional headaches, denies chest pain or shortness of breath. States he has been taking medications as prescribed. O: BP 138/88 mmHg (right arm, seated), HR 74 bpm, SpO2 98%. Weight 88.5 kg. A: Hypertension - adequately controlled on current regimen. Diabetes - HbA1c slightly elevated, adjust diet counseling. P: Continue Lisinopril 10mg QD. Recheck BP in 4 weeks via RPM. Patient instructed on low-sodium diet.
Visit note 2025-07-02: S: Follow-up

## 2. Chunking Strategy

**Config:** `chunk_size=512`, `overlap=64` — balances context window cost against retrieval granularity.

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from rag.config import CHUNK_SIZE, CHUNK_OVERLAP

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_text(text)

print(f"Chunk size: {CHUNK_SIZE} chars | Overlap: {CHUNK_OVERLAP} chars")
print(f"Chunks produced for PT-001: {len(chunks)}\n")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print()

Chunk size: 512 chars | Overlap: 64 chars
Chunks produced for PT-001: 3

--- Chunk 0 (403 chars) ---
Patient: Robert Harmon, age 64, sex M
Problem list: Essential hypertension (I10); Type 2 diabetes mellitus (E11.9); Hyperlipidemia (E78.5)
Medications: Lisinopril 10mg QD; Metformin 500mg BID; Atorvastatin 20mg QHS
Allergies: Penicillin - Urticaria
Lab BMP on 2025-10-15: Na=139, K=4.2, Cr=1.0, BUN=18, Glucose=112
Lab HbA1c on 2025-10-15: HbA1c=7.1
Lab Lipid panel on 2025-10-15: LDL=98, HDL=45, TG=142

--- Chunk 1 (488 chars) ---
Visit note 2025-10-15: S: Patient presents for routine follow-up. Reports occasional headaches, denies chest pain or shortness of breath. States he has been taking medications as prescribed. O: BP 138/88 mmHg (right arm, seated), HR 74 bpm, SpO2 98%. Weight 88.5 kg. A: Hypertension - adequately controlled on current regimen. Diabetes - HbA1c slightly elevated, adjust diet counseling. P: Continue Lisinopril 10mg QD. Recheck BP in 4 weeks via RPM. Patient instruct

## 3. Ingest All Patients into ChromaDB

In [4]:
from rag.ingest import ingest_all

VECTORSTORE = "../vectorstore_demo"
print("Ingesting all 12 patient EHR records...")
ingest_all(data_dir="../data/patients", persist_dir=VECTORSTORE)
print("Done.")

Ingesting all 12 patient EHR records...


Ingested PT-001


Ingested PT-002


Ingested PT-003


Ingested PT-004


...

Ingested PT-008


Ingested PT-009


Ingested PT-010


Ingested PT-011


Ingested PT-012
Done.


## 4. Retrieval — Clinical Queries

In [5]:
from rag.retriever import retrieve

query = "blood pressure history and antihypertensive medications"
results = retrieve("PT-001", query, k=3, persist_dir=VECTORSTORE)

print(f"Query: '{query}'\n")
print(f"Patient: PT-001 | k=3\n")
for i, r in enumerate(results):
    print(f"--- Result {i+1} (score: {r['score']:.3f}, chunk: {r['chunk_id']}) ---")
    print(r["content"])
    print()

Query: 'blood pressure history and antihypertensive medications'

Patient: PT-001 | k=3

--- Result 1 (score: 0.360, chunk: 1) ---
Visit note 2025-10-15: S: Patient presents for routine follow-up. Reports occasional headaches, denies chest pain or shortness of breath. States he has been taking medications as prescribed. O: BP 138/88 mmHg (right arm, seated), HR 74 bpm, SpO2 98%. Weight 88.5 kg. A: Hypertension - adequately controlled on current regimen. Diabetes - HbA1c slightly elevated, adjust diet counseling. P: Continue Lisinopril 10mg QD. Recheck BP in 4 weeks via RPM. Patient instructed on low-sodium diet.

--- Result 2 (score: 0.301, chunk: 0) ---
Patient: Robert Harmon, age 64, sex M
Problem list: Essential hypertension (I10); Type 2 diabetes mellitus (E11.9); Hyperlipidemia (E78.5)
Medications: Lisinopril 10mg QD; Metformin 500mg BID; Atorvastatin 20mg QHS
Allergies: Penicillin - Urticaria
Lab BMP on 2025-10-15: Na=139, K=4.2, Cr=1.0, BUN=18, Glucose=112
Lab HbA1c on 2025-10-1

In [6]:
# Query 2: ACE inhibitor cough — should surface the July 2025 visit note
query2 = "cough side effects medication"
results2 = retrieve("PT-001", query2, k=2, persist_dir=VECTORSTORE)

print(f"Query: '{query2}'")
for r in results2:
    print(f"\nscore={r['score']:.3f} chunk={r['chunk_id']}")
    print(r["content"])

Query: 'cough side effects medication'

score=0.188 chunk=2
Visit note 2025-07-02: S: Follow-up for hypertension management. Patient complains of dry, persistent cough for the past 3 weeks. Denies fever or URI symptoms. Reports cough started about 2 weeks after last Lisinopril dose increase. O: BP 142/90, HR 78. Lungs clear. A: Likely ACE inhibitor-induced cough secondary to Lisinopril. P: Discussed switching to ARB if cough persists. Patient prefers to continue Lisinopril for now.

score=0.003 chunk=1
Visit note 2025-10-15: S: Patient presents for routine follow-up. Reports occasional headaches, denies chest pain or shortness of breath. States he has been taking medications as prescribed. O: BP 138/88 mmHg (right arm, seated), HR 74 bpm, SpO2 98%. Weight 88.5 kg. A: Hypertension - adequately controlled on current regimen. Diabetes - HbA1c slightly elevated, adjust diet counseling. P: Continue Lisinopril 10mg QD. Recheck BP in 4 weeks via RPM. Patient instructed on low-sodium diet.


## 5. Retrieval Quality — Precision & Recall

For the missed_medication scenario, we know which chunks are clinically relevant. We measure how well the retriever surfaces them.

In [7]:
from evaluation.metrics import retrieval_precision, retrieval_recall

# Ground truth: for missed_medication, chunk 0 (medications + problem list) and
# chunk 1 (visit notes mentioning cough) are the relevant ones.
# Adjust based on actual chunk breakdown shown above.
relevant_chunks = [0, 1]

query_mm = "blood pressure medication adherence cough side effect"
retrieved = retrieve("PT-001", query_mm, k=5, persist_dir=VECTORSTORE)
retrieved_ids = [r["chunk_id"] for r in retrieved]

precision = retrieval_precision(retrieved_ids, relevant_chunks)
recall = retrieval_recall(retrieved_ids, relevant_chunks)

print(f"Retrieved chunk IDs: {retrieved_ids}")
print(f"Relevant chunk IDs:  {relevant_chunks}")
print(f"\nPrecision: {precision:.2f}  (target ≥ 0.80)")
print(f"Recall:    {recall:.2f}  (target ≥ 0.75)")

Retrieved chunk IDs: [2, 1, 0]
Relevant chunk IDs:  [0, 1]

Precision: 0.67  (target ≥ 0.80)
Recall:    1.00  (target ≥ 0.75)


## 6. Cross-Patient Isolation Check

The retriever is scoped to a single patient's collection. A query for PT-001 must never return PT-002 data.

In [8]:
# PT-002 has heart failure; PT-001 does not. Querying PT-001 for HF terms should return low scores.
hf_query = "heart failure fluid retention ejection fraction BNP"

pt001_results = retrieve("PT-001", hf_query, k=3, persist_dir=VECTORSTORE)
pt008_results = retrieve("PT-008", hf_query, k=3, persist_dir=VECTORSTORE)

print("PT-001 scores for heart failure query (should be low):")
for r in pt001_results:
    print(f"  score={r['score']:.3f}")

print("\nPT-008 scores for heart failure query (should be high):")
for r in pt008_results:
    print(f"  score={r['score']:.3f}")

PT-001 scores for heart failure query (should be low):
  score=0.206
  score=0.135
  score=0.130

PT-008 scores for heart failure query (should be high):
  score=0.404
  score=0.342
  score=0.331


## 7. Embedding Model Summary

| Parameter | Value |
|---|---|
| Model | `text-embedding-3-small` |
| Chunk size | 512 chars |
| Chunk overlap | 64 chars |
| Retrieval k | 5 |
| Vector store | ChromaDB (local) |
| Collection naming | `ehr_{patient_id}` |

The overlap ensures that sentences split across chunk boundaries still have retrieval coverage.